#  Colab: scalable filing-level pipeline

We decided to work on financial data by asking the question, is it possible to identify the financial distress (negative marjet reaction) of a firm through its 10-k files?


1. Streams firm-level SEC 10-K records from Hugging Face JSONL shards.
2. Builds filing-level rows incrementally — it never stores millions of sentences.
3. Runs classical TF-IDF experiments across sections and horizons.
4. Generates ROC/PR curves and a confusion matrix.
5. Writes `report_summary.md`, CSVs, and PNGs under `/content/financial_distress_outputs/`.

CPU-safe by default. Increase shard/firm counts for more data; Colab Pro/ GPU is not required for the classical pipeline.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-pptx", "pyarrow"])

import json
import itertools
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, auc, precision_recall_curve,
    ConfusionMatrixDisplay,
)
from sklearn.svm import LinearSVC

DATASET = "JanosAudran/financial-reports-sec"
REVISION = "main"
OUT = Path("/content/financial_distress_outputs")
FIG = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# Moderate scale-up for Colab CPU/RAM; Item 2 is excluded from standalone ablation.
CONFIG = {
    "train":     {"shards": list(range(0, 12)), "firms_per_shard": 18},
    "validate":  {"shards": list(range(0, 8)),  "firms_per_shard": 10},
    "test":      {"shards": list(range(0, 12)), "firms_per_shard": 12},
}
KEY_SECTIONS = ["1A", "7", "8", "1"]
HORIZONS = ["1d", "5d", "30d"]
MIN_SENTENCES = 10
LABEL_MAP = {"positive": 0, "negative": 1}
SECTION_KEYS = [
    "section_1", "section_1A", "section_1B", "section_2", "section_3",
    "section_4", "section_5", "section_6", "section_7", "section_7A",
    "section_8", "section_9", "section_9A", "section_9B", "section_10",
    "section_11", "section_12", "section_13", "section_14", "section_15",
]
SECTION_TO_CODE = {k: i for i, k in enumerate(SECTION_KEYS)}
COLS = [
    "docID", "section_name", "cik", "name", "filingDate", "split",
    "label_1d", "label_5d", "label_30d",
    "text", "n_sentences",
]
print("Output dir:", OUT)
print("Config:", CONFIG)
print("Key sections:", KEY_SECTIONS)

Output dir: /content/financial_distress_outputs
Config: {'train': {'shards': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 'firms_per_shard': 18}, 'validate': {'shards': [0, 1, 2, 3, 4, 5, 6, 7], 'firms_per_shard': 10}, 'test': {'shards': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 'firms_per_shard': 12}}
Key sections: ['1A', '7', '8', '1']


## 1. Build filing-level dataset from raw firm JSONL

In [2]:
import re

# Clean text by removing HTML tags, newlines, tabs, carriage returns, dates, fiscal years, and non-alphanumeric characters (except for some punctuation). Normalize whitespace and convert to lowercase.
def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\\n|\\t|\\r", " ", text)
    text = re.sub(r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}\b', ' ', text)
    text = re.sub(r'\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?,?\s+\d{4}\b', ' ', text)
    text = re.sub(r'\b\d{4}-\d{2}-\d{2}\b', ' ', text)
    text = re.sub(r'\b(?:fiscal\s+)?(?:year|20\d{2}|19\d{2})\s+(?:20\d{2}|19\d{2})\b', ' ', text)
    text = re.sub(r'\b(?:FISCAL\s+)?(?:year|20\d{2}|19\d{2})\s+(?:20\d{2}|19\d{2})\b', ' ', text)
    text = re.sub(r'\b(?:19|20)\d{2}\b', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s\.\$%\-\(\)]', ' ', text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

In [3]:
# Data pipeline that streams K-10 filings from the Hugging Face dataset, extracts relevant information, and writes them to a parquet file.
# build the URL for a given split (train, test, validate) and shard ID
def shard_url(split: str, shard_id: int) -> str:
    return f"https://huggingface.co/datasets/{DATASET}/resolve/{REVISION}/data/large/{split}/shard_{shard_id}.jsonl"


# chunk 8 MB at a time, yielding lines from the response
def iter_lines(response, chunk_size=8 * 1024 * 1024):
    buffer = b""
    for chunk in response.iter_content(chunk_size=chunk_size):
        buffer += chunk
        while b"\n" in buffer:
            line, buffer = buffer.split(b"\n", 1)
            if line:
                yield line
    if buffer:
        yield buffer

# stream firms from the specified shards (iterate through shard_ids), yielding one firm at a time
def stream_firms(split: str, shard_ids, per_shard: int):
    for shard_id in shard_ids:
        url = shard_url(split, shard_id)
        taken = 0
        print(f"Reading {split} shard {shard_id}", flush=True)
        try:
            with requests.get(url, stream=True, timeout=120) as response:
                if response.status_code == 404:
                    print("  -> shard not found, skipping", flush=True)
                    continue
                response.raise_for_status()
                for raw in iter_lines(response):
                    yield json.loads(raw)
                    taken += 1
                    if taken >= per_shard:
                        break
        except Exception as e:
            print(f"  -> shard {shard_id} failed: {e}", flush=True)
            continue
        print(f"  -> took {taken} firms", flush=True)
        if taken < per_shard:
            print(f"  warning: shard {shard_id} had fewer than {per_shard} firms", flush=True)

# label a given set of labels for a specific horizon
def _label(labels: dict, horizon: str):
    v = labels.get(horizon)
    return LABEL_MAP.get(v, v)

# convert a firm dictionary to a list of filing rows, one per section (and one for "ALL" sections combined)
def firm_to_filing_rows(firm: dict, split: str) -> list[dict]:
    rows = []
    cik = firm.get("cik")
    name = firm.get("name")
    for filing in firm.get("filings", []):
        form = filing.get("form", "")
        report_year = str(filing.get("reportDate") or "")[:4]
        doc_id = f"{cik}_{form}_{report_year}"
        labels = {k: LABEL_MAP.get(v, v) for k, v in filing.get("labels", {}).items()}

        section_texts = {}
        section_counts = {}
        for skey, sentences in filing.get("report", {}).items():
            if not sentences:
                continue
            sname = skey.replace("section_", "")
            kept = [s for s in sentences if isinstance(s, str) and len(s) > 10] # filter out short sentences
            if not kept:
                continue
            text = clean_text(" ".join(kept)) # clean and join sentences
            if len(text) < 10: # too short after cleaning
                continue
            section_texts[sname] = text # text for this section
            section_counts[sname] = len(kept) # number of sentences for this section

        if not section_texts:
            continue

        base = {
            "docID": doc_id,
            "cik": cik,
            "name": name,
            "filingDate": filing.get("filingDate"),
            "split": split,
            "label_1d": _label(labels, "1d"),
            "label_5d": _label(labels, "5d"),
            "label_30d": _label(labels, "30d"),
        }

        rows.append({**base, "section_name": "ALL",
                     "text": " ".join(section_texts.values()),
                     "n_sentences": sum(section_counts.values())})
        for sec in KEY_SECTIONS:
            if sec in section_texts:
                rows.append({**base, "section_name": sec,
                             "text": section_texts[sec],
                             "n_sentences": section_counts[sec]})
    return rows

# Writer class to write filings to a parquet file in batches
class FilingWriter:
    def __init__(self, path: Path):
        self.path = path
        self.buf = []
        self.writer = None
    def add(self, rows):
        self.buf.extend(rows)
        if len(self.buf) >= 1000:
            self.flush()
    def flush(self):
        if not self.buf:
            return
        table = pa.Table.from_pylist(self.buf)
        if self.writer is None:
            self.writer = pq.ParquetWriter(self.path, table.schema, compression="snappy")
        self.writer.write_table(table)
        self.buf = []
    def close(self):
        self.flush()
        if self.writer is not None:
            self.writer.close()

# orchestration function to build a split (train, validate, test) by streaming firms from the specified shards and writing them to a parquet file
def build_split(split: str, shards, firms_per_shard: int) -> Path:
    path = OUT / f"filing_{split}.parquet"
    writer = FilingWriter(path)
    n_firms = 0
    for firm in stream_firms(split, shards, firms_per_shard):
        rows = firm_to_filing_rows(firm, split)
        if rows:
            writer.add(rows)
            n_firms += 1
        if n_firms and n_firms % 20 == 0:
            print(f"  {split}: {n_firms} firms with filings processed", flush=True)
    writer.close()
    print(f"Wrote {path}", flush=True)
    return path

# Build all splits (train, validate, test) according to the CONFIG dictionary
for split, cfg in CONFIG.items():
    build_split(split, cfg["shards"], cfg["firms_per_shard"])

Reading train shard 0
  -> took 18 firms
Reading train shard 1
  train: 20 firms with filings processed
  -> took 18 firms
Reading train shard 2
  train: 40 firms with filings processed
  -> took 18 firms
Reading train shard 3
  train: 60 firms with filings processed
  -> took 18 firms
Reading train shard 4
  train: 80 firms with filings processed
  -> took 18 firms
Reading train shard 5
  train: 100 firms with filings processed
  -> took 18 firms
Reading train shard 6
  train: 120 firms with filings processed
  -> took 18 firms
Reading train shard 7
  train: 140 firms with filings processed
  -> took 18 firms
Reading train shard 8
  train: 160 firms with filings processed
  -> took 18 firms
Reading train shard 9
  train: 180 firms with filings processed
  -> took 18 firms
Reading train shard 10
  -> shard not found, skipping
Reading train shard 11
  -> shard not found, skipping
Wrote /content/financial_distress_outputs/filing_train.parquet
Reading validate shard 0
  -> took 10 firms
R

## 2. Load filing-level splits

In [4]:
# load a parquet format dataset for a given split, or return an empty dataframe if missing
def load_split(name):
    path = OUT / f"filing_{name}.parquet"
    if not path.exists():
        print("Missing", path)
        return pd.DataFrame(columns=COLS)
    return pd.read_parquet(path)

train_f = load_split("train")
val_f = load_split("validate")
test_f = load_split("test")

# delete unnecessary columns and filter out filings with too few sentences
for d in (train_f, val_f, test_f):
    if len(d):
        d.drop(columns=[c for c in d.columns if c not in COLS], inplace=True, errors="ignore")
        mask = d["n_sentences"] >= MIN_SENTENCES
        d.drop(index=d.index[~mask], inplace=True)
    d.reset_index(drop=True, inplace=True)

# Print counts after filtering
print("Counts after n_sentences >=", MIN_SENTENCES)
for name, d in [("train", train_f), ("validate", val_f), ("test", test_f)]:
    print(name, len(d), "rows,", d["docID"].nunique() if len(d) else 0, "filings,", d["cik"].nunique() if len(d) else 0, "firms")
if len(train_f):
    print(train_f.groupby("section_name").size().sort_values(ascending=False).head(10))

# Compute label prevalence for each section, horizon, and split, and save to CSV
prev_rows = []
for sec in ["ALL"] + KEY_SECTIONS:
    for h in HORIZONS:
        lab = f"label_{h}"
        for split_name, d in [("validation", val_f), ("test", test_f)]:
            sub = d[d["section_name"] == sec]
            if len(sub):
                prev_rows.append({
                    "section": sec, "horizon": h, "eval_split": split_name,
                    "n_eval": len(sub), "firms": sub["cik"].nunique(),
                    "pos_rate": float(sub[lab].mean()),
                })
pd.DataFrame(prev_rows).to_csv(OUT / "label_prevalence.csv", index=False)
print("Saved label_prevalence.csv")

Counts after n_sentences >= 10
train 14764 rows, 3813 filings, 180 firms
validate 1029 rows, 233 filings, 80 firms
test 2583 rows, 575 filings, 120 firms
section_name
ALL    3813
1      3506
7      2972
1A     2299
8      2174
dtype: int64
Saved label_prevalence.csv


## 3. Models and metrics

In [5]:
# metrics accuracy, precision, recall, f1, roc_auc, pr_auc
def metrics_dict(y_true, y_pred, y_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
    }

# train the model on train_df, predict on eval_df, and return predictions and scores
def fit_predict(pipe, train_df, eval_df, label_col):
    pipe.fit(train_df["text"], train_df[label_col])
    y_pred = pipe.predict(eval_df["text"])
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(eval_df["text"])[:, 1]
    else:
        y_score = pipe.decision_function(eval_df["text"])
    return y_pred, y_score

# after training the model, evaluate it on eval_df and return metrics
def evaluate(pipe, train_df, eval_df, label_col):
    y_pred, y_score = fit_predict(pipe, train_df, eval_df, label_col)
    m = metrics_dict(eval_df[label_col], y_pred, y_score)
    m.update({"n_train": len(train_df), "n_eval": len(eval_df)})
    return m

# build a TF-IDF vectorizer with optional overrides for parameters
def tfidf(**overrides):
    p = dict(max_features=10000, ngram_range=(1, 2), stop_words="english", min_df=2)
    p.update(overrides)
    return TfidfVectorizer(**p)

# build a model pipeline based on the specified kind
def make_model(kind):
    improved = dict(max_features=20000, min_df=3, max_df=0.9, sublinear_tf=True)
    if kind == "base_logreg":
        return Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    if kind == "improved_logreg":
        return Pipeline([("tfidf", tfidf(**improved)), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    if kind == "unbalanced_logreg":
        return Pipeline([("tfidf", tfidf(**improved)), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
    if kind == "complement_nb":
        return Pipeline([("tfidf", tfidf(**improved, norm=None)), ("clf", ComplementNB())])
    if kind == "linearsvc":
        return Pipeline([("tfidf", tfidf(**improved)), ("clf", LinearSVC(class_weight="balanced", max_iter=5000, random_state=42))])
    raise ValueError(kind)

MODEL_KINDS = ["base_logreg", "improved_logreg", "unbalanced_logreg", "complement_nb", "linearsvc"]

# return the train, validation, and test dataframes for a given section
def section_dfs(section):
    return (train_f[train_f.section_name == section],
            val_f[val_f.section_name == section],
            test_f[test_f.section_name == section])

## 4. Model comparison on ALL / 30d

In [7]:
# build the model for the "ALL" section and the 30-day horizon, evaluate on validation and test splits, and save results
rows = []
tr, va, te = section_dfs("ALL")
for kind in MODEL_KINDS:
    for split_name, ev in [("validation", va), ("test", te)]:
        m = evaluate(make_model(kind), tr, ev, "label_30d")
        rows.append({"model": kind, "section": "ALL", "horizon": "30d", "eval_split": split_name, **m})
model_results = pd.DataFrame(rows)
model_results.to_csv(OUT / "classical_model_results.csv", index=False)
print(model_results.sort_values(["eval_split", "pr_auc"], ascending=[True, False]).to_string(index=False))

best_kind = (model_results[(model_results.eval_split == "validation") & (model_results.n_eval >= 20)]
             .sort_values("pr_auc", ascending=False).iloc[0]["model"])
print("\nBest model by validation PR-AUC:", best_kind)

            model section horizon eval_split  accuracy  precision   recall       f1  roc_auc   pr_auc  n_train  n_eval
  improved_logreg     ALL     30d       test  0.577391   0.570815 0.860841 0.686452 0.598012 0.627991     3813     575
unbalanced_logreg     ALL     30d       test  0.577391   0.597633 0.653722 0.624420 0.597988 0.627499     3813     575
        linearsvc     ALL     30d       test  0.553043   0.562201 0.760518 0.646492 0.584264 0.617596     3813     575
      base_logreg     ALL     30d       test  0.558261   0.556468 0.877023 0.680905 0.550758 0.597683     3813     575
    complement_nb     ALL     30d       test  0.551304   0.549902 0.909385 0.685366 0.535392 0.556846     3813     575
        linearsvc     ALL     30d validation  0.635193   0.649123 0.816176 0.723127 0.628563 0.691677     3813     233
      base_logreg     ALL     30d validation  0.575107   0.593909 0.860294 0.702703 0.568375 0.656337     3813     233
  improved_logreg     ALL     30d validation  0.

## 5. Section ablation on 30d

In [8]:
sections = [s for s in ["ALL"] + KEY_SECTIONS if s in set(train_f.section_name) | set(val_f.section_name) | set(test_f.section_name)]
srows = []
for sec in sections:
    tr, va, te = section_dfs(sec)
    if len(tr) < 10 or len(te) < 5:
        continue
    for split_name, ev in [("validation", va), ("test", te)]:
        m = evaluate(make_model(best_kind), tr, ev, "label_30d")
        srows.append({"section": sec, "eval_split": split_name, **m})
section_results = pd.DataFrame(srows)
section_results.to_csv(OUT / "section_results_30d.csv", index=False)
print(section_results[section_results.eval_split == "test"].sort_values("pr_auc", ascending=False).to_string(index=False))

leaders = (section_results[(section_results.eval_split == "validation") & (section_results.n_eval >= 20)]
           .sort_values("pr_auc", ascending=False))
best_section = leaders.iloc[0]["section"] if len(leaders) else "ALL"
horizon_sections = ["ALL"] + [s for s in leaders["section"].tolist() if s != "ALL"][:2]
horizon_sections = list(dict.fromkeys(horizon_sections))
print("\nValidation section leaders (n_eval>=20):")
print(leaders[["section", "pr_auc", "f1", "n_eval"]].to_string(index=False))
print("Best section:", best_section)
print("Horizon sections:", horizon_sections)

section eval_split  accuracy  precision   recall       f1  roc_auc   pr_auc  n_train  n_eval
    ALL       test  0.553043   0.562201 0.760518 0.646492 0.584264 0.617596     3813     575
      8       test  0.557029   0.576037 0.625000 0.599520 0.571751 0.583823     2174     377
      1       test  0.533095   0.554180 0.604730 0.578352 0.534368 0.570803     3506     559
      7       test  0.515679   0.541209 0.639610 0.586310 0.500690 0.558477     2972     574
     1A       test  0.508032   0.514354 0.836576 0.637037 0.532670 0.542238     2299     498

Validation section leaders (n_eval>=20):
section   pr_auc       f1  n_eval
     1A 0.703002 0.720539     206
    ALL 0.691677 0.723127     233
      1 0.642180 0.610687     226
      8 0.633254 0.619355     132
      7 0.607045 0.604982     232
Best section: 1A
Horizon sections: ['ALL', '1A', '1']


## 6. Horizon sweep

In [9]:
hrows = []
for sec in horizon_sections:
    tr, va, te = section_dfs(sec)
    for h in HORIZONS:
        lab = f"label_{h}"
        for split_name, ev in [("validation", va), ("test", te)]:
            m = evaluate(make_model(best_kind), tr, ev, lab)
            hrows.append({"section": sec, "horizon": h, "eval_split": split_name, **m})
horizon_results = pd.DataFrame(hrows)
horizon_results.to_csv(OUT / "horizon_results.csv", index=False)
print(horizon_results[horizon_results.eval_split == "test"].sort_values("pr_auc", ascending=False).head(15).to_string(index=False))

section horizon eval_split  accuracy  precision   recall       f1  roc_auc   pr_auc  n_train  n_eval
    ALL      1d       test  0.566957   0.595376 0.653968 0.623298 0.607473 0.669270     3813     575
    ALL      5d       test  0.568696   0.602305 0.655172 0.627628 0.591864 0.661663     3813     575
      1      1d       test  0.549195   0.584665 0.600000 0.592233 0.570156 0.648513     3506     559
    ALL     30d       test  0.553043   0.562201 0.760518 0.646492 0.584264 0.617596     3813     575
     1A      1d       test  0.538153   0.537838 0.771318 0.633758 0.567280 0.584270     2299     498
     1A      5d       test  0.524096   0.531977 0.706564 0.606965 0.543804 0.574744     2299     498
      1     30d       test  0.533095   0.554180 0.604730 0.578352 0.534368 0.570803     3506     559
      1      5d       test  0.515206   0.551247 0.646104 0.594918 0.503163 0.558724     3506     559
     1A     30d       test  0.508032   0.514354 0.836576 0.637037 0.532670 0.542238     229

## 7. ROC / PR curves and confusion matrix

In [15]:
def fit_score(kind, tr, ev, label_col):
    pipe = make_model(kind)
    pipe.fit(tr["text"], tr[label_col])
    if hasattr(pipe, "predict_proba"):
        s = pipe.predict_proba(ev["text"])[:, 1]
    else:
        s = pipe.decision_function(ev["text"])
    return pipe, s

def plot_roc_pr(y_true, y_score, title, path):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.3f}")
    ax[0].plot([0, 1], [0, 1], "--", color="gray")
    ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate")
    ax[0].set_title(f"ROC — {title}"); ax[0].legend()
    ax[1].plot(rec, prec, label=f"AP={average_precision_score(y_true, y_score):.3f}")
    ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
    ax[1].set_title(f"PR — {title}"); ax[1].legend()
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)

available_sections = set(train_f.section_name) | set(val_f.section_name) | set(test_f.section_name)
fig_specs = [
    ("ALL", "30d", "ALL sections — 30d", FIG / "roc_pr_ALL_30d.png"),
    ("ALL", "5d",  "ALL sections — 5d",  FIG / "roc_pr_ALL_5d.png"),
    ("ALL", "1d",  "ALL sections — 1d",  FIG / "roc_pr_ALL_1d.png"),
    ("1", "1d",  "1 section — 1d",  FIG / "roc_pr_1_1d.png"),
    ("1A", "1d",  "1A section — 1d",  FIG / "roc_pr_1A_1d.png"),
]

candidates = []
if best_section != "ALL":
    candidates.append(best_section)
if len(horizon_sections) > 1:
    candidates.append(horizon_sections[1])
if "7" in available_sections:
    candidates.append("7")

sec_fig = None
for cand in candidates:
    if cand in available_sections and cand != "ALL":
        fig_specs.append((cand, "30d", f"{cand} — 30d", FIG / f"roc_pr_{cand}_30d.png"))
        sec_fig = cand
        break

for sec, h, title, path in fig_specs:
    tr, va, te = section_dfs(sec)
    if len(tr) < 10 or len(te) < 5:
        continue
    lab = f"label_{h}"
    pipe, y_score = fit_score(best_kind, tr, te, lab)
    plot_roc_pr(te[lab].to_numpy(), y_score, f"{title} (test)", path)
    print("Saved", path)
    if sec == "ALL" or h == "1d":
        y_pred = pipe.predict(te["text"])
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(te[lab], y_pred, ax=ax, cmap="Blues")
        ax.set_title(f"Confusion matrix — {sec} / {h}")
        fig.tight_layout(); fig.savefig(FIG / f"confusion_{sec}_{h}.png", dpi=160); plt.close(fig)
        print("Saved", FIG / f"confusion_{sec}_{h}.png")

Saved /content/financial_distress_outputs/figures/roc_pr_ALL_30d.png
Saved /content/financial_distress_outputs/figures/confusion_ALL_30d.png
Saved /content/financial_distress_outputs/figures/roc_pr_ALL_5d.png
Saved /content/financial_distress_outputs/figures/confusion_ALL_5d.png
Saved /content/financial_distress_outputs/figures/roc_pr_ALL_1d.png
Saved /content/financial_distress_outputs/figures/confusion_ALL_1d.png
Saved /content/financial_distress_outputs/figures/roc_pr_1_1d.png
Saved /content/financial_distress_outputs/figures/confusion_1_1d.png
Saved /content/financial_distress_outputs/figures/roc_pr_1A_1d.png
Saved /content/financial_distress_outputs/figures/confusion_1A_1d.png
Saved /content/financial_distress_outputs/figures/roc_pr_1A_30d.png


## 9. Download outputs

In [16]:
from pathlib import Path
try:
    from google.colab import files
    for p in [OUT / "report_summary.md"]:
        print("Downloading", p)
        files.download(str(p))
except Exception as e:
    print("Not in Colab or download skipped:", e)
print("All outputs:", OUT)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(p)

Not in Colab or download skipped: Cannot find file: /content/financial_distress_outputs/report_summary.md
All outputs: /content/financial_distress_outputs
/content/financial_distress_outputs/classical_model_results.csv
/content/financial_distress_outputs/figures/confusion_1A_1d.png
/content/financial_distress_outputs/figures/confusion_1_1d.png
/content/financial_distress_outputs/figures/confusion_ALL_1d.png
/content/financial_distress_outputs/figures/confusion_ALL_30d.png
/content/financial_distress_outputs/figures/confusion_ALL_5d.png
/content/financial_distress_outputs/figures/roc_pr_1A_1d.png
/content/financial_distress_outputs/figures/roc_pr_1A_30d.png
/content/financial_distress_outputs/figures/roc_pr_1_1d.png
/content/financial_distress_outputs/figures/roc_pr_ALL_1d.png
/content/financial_distress_outputs/figures/roc_pr_ALL_30d.png
/content/financial_distress_outputs/figures/roc_pr_ALL_5d.png
/content/financial_distress_outputs/filing_test.parquet
/content/financial_distress_outp

In [17]:
# Create a zip file named folder_name.zip containing the folder
!zip -r financial_name_final_27.zip financial_distress_outputs/

# Download the resulting zip file
from google.colab import files
files.download('financial_name_final_27.zip')

  adding: financial_distress_outputs/ (stored 0%)
  adding: financial_distress_outputs/figures/ (stored 0%)
  adding: financial_distress_outputs/figures/confusion_ALL_1d.png (deflated 15%)
  adding: financial_distress_outputs/figures/roc_pr_1A_30d.png (deflated 14%)
  adding: financial_distress_outputs/figures/roc_pr_1_1d.png (deflated 12%)
  adding: financial_distress_outputs/figures/confusion_1A_1d.png (deflated 16%)
  adding: financial_distress_outputs/figures/roc_pr_ALL_30d.png (deflated 12%)
  adding: financial_distress_outputs/figures/roc_pr_1A_1d.png (deflated 13%)
  adding: financial_distress_outputs/figures/confusion_ALL_30d.png (deflated 16%)
  adding: financial_distress_outputs/figures/confusion_ALL_5d.png (deflated 15%)
  adding: financial_distress_outputs/figures/confusion_1_1d.png (deflated 16%)
  adding: financial_distress_outputs/figures/roc_pr_ALL_1d.png (deflated 13%)
  adding: financial_distress_outputs/figures/roc_pr_ALL_5d.png (deflated 13%)
  adding: financial_dis

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>